# Curriculum 04 · Lab 2 — MMR: trading relevance for diversity

**Goal:** Fix the failure mode of plain top-k: when the k closest passages
are near-duplicates (same article, same facts), the context you hand the LLM
is redundant. Maximum Marginal Relevance (MMR) re-ranks the candidate pool to
trade a little relevance for diversity:

```
mmr_score(doc) = lambda * relevance(doc)
                 - (1 - lambda) * max_sim(doc, already_selected)

lambda_mult = 1.0 -> pure relevance: identical to plain similarity search
lambda_mult = 0.5 -> balanced: relevant but not redundant (the usual default)
lambda_mult = 0.0 -> pure diversity: farthest-point sampling from the query
```

```
Retriever   : MMRRetriever (retrieval/mmr.py) over FAISSVectorStore
Measures    : overlap with plain top-k (relevance proxy) and
              mean pairwise 1-cosine distance (diversity proxy)
Embedding   : BGE (BAAI/bge-base-en-v1.5, local, CPU)
Data        : rag-mini-wikipedia (first 100 passages, questions 1606/1610/1604)
```

**Why MMR:** a diverse top-5 often beats five near-duplicates of the same
fact inside a RAG context window — the LLM gets more distinct evidence per
token. But pure diversity drifts off-topic, which is exactly why the lab
sweeps all three lambdas and makes the tradeoff measurable instead of
hand-wavy.

This is the second lab of track 04-retrieval (see
`.omo/plans/layer1-rag-playbook.md`).


## 0 · Setup — environment, imports & repo paths

**WHAT:** Installs the lab's dependencies (a no-op if already present),
imports pandas plus the repo's retriever/store classes and BGE embedder, and
puts the repo-root component library on `sys.path` so this notebook reuses
`src/retrieval/*.py`, `src/vectordb/faiss.py` and `src/embeddings/bge.py` exactly like the
lab script.

**WHY:** Everything embeds **locally** with BGE via sentence-transformers —
no API embeddings anywhere. The retriever classes live in the repo's shared
component library (`src/retrieval/`), not inside the lab, so the exact same code
path runs here, in the `.py`, and in later tracks.

**Paths:** the next cell resolves the **repo root** automatically — it works
whether the kernel launches from the repo root (like the lab script) or from
the notebook's own folder (the Jupyter default) — and `cd`s into it so every
path stays repo-relative.

**WHAT TO EXPECT:** no output from the pip cell (packages already
installed), a silent import from the second. The BGE model is loaded lazily
when the experiment cell first calls it.


In [1]:
# Lab-specific dependencies (already in requirements.txt — the install is a
# no-op safety net for fresh environments):
#   sentence-transformers -> local BGE embeddings (embeddings/bge.py)
#   faiss-cpu             -> the FAISS index (vectordb/faiss.py)
#   pandas                -> reads the passages/test.parquet corpus
%pip install sentence-transformers faiss-cpu pandas



[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

import pandas as pd

# Make the repo-root component library importable. A notebook has no
# ``__file__``, so resolve the repo root by walking up from the kernel's
# working directory — this works whether the kernel launches from the repo
# root (like the lab script) or from the notebook's own folder (Jupyter's
# default) — then cd into it so every repo-relative path behaves exactly
# like the .py.
REPO_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "curriculum").is_dir() and (candidate / "NoteBooks").is_dir():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

from embeddings.bge import BGEEmbedding  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from retrieval.mmr import MMRRetriever  # noqa: E402
from vectordb.faiss import FAISSVectorStore  # noqa: E402


## 1 · Configuration — the experiment's knobs

**WHAT:** The same corpus/embedding constants as lab 01 plus the MMR-specific
ones: `TOP_K = 5` (plain similarity hits), `MMR_K = 5` (same budget, different
ranking), and `LAMBDAS = (1.0, 0.5, 0.0)` — the relevance-vs-diversity sweep.

**WHY:** Keeping the plain and MMR budgets identical (`TOP_K == MMR_K`) makes
the comparison apples-to-apples: the only thing that changes between the two
columns is the ranking rule, not the number of documents.


In [3]:
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
TEST_PATH = Path("Data/corpus/rag-mini-wikipedia/test.parquet")
N_PASSAGES = 100  # deterministic head of the 3200-passage corpus (keeps runtime low)
QUESTION_IDS = [1606, 1610, 1604]  # real questions from test.parquet, answers inside the subset
TOP_K = 5  # plain similarity search returns this many hits
MMR_K = 5  # MMR returns this many hits (same budget, different ranking)
LAMBDAS = (1.0, 0.5, 0.0)  # MMR sweep: 1.0 = pure relevance, 0.0 = pure diversity
PREVIEW = 62  # max characters of passage text shown next to each hit
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
BGE_DIM = 768


## 2 · Load — corpus + questions + the two measurement helpers

**WHAT:** The same corpus/format helpers as lab 01, plus `passage_lookup`
(id -> text) and `mean_pairwise_cosine_distance` — the lab's diversity proxy:
mean `1 - cosine` over every pair of returned passage vectors (BGE vectors are
L2-normalized, so cosine is a plain dot product and `1 - cosine` is a proper
distance in [0, 2]).

**WHY:** Every lab in this track prints two numbers per retrieval: *what*
came back (the ids) and *how good* it is (a score). For MMR those two numbers
are the overlap with plain top-k (relevance) and this mean distance (spread) —
the pair makes the tradeoff visible in one line per lambda.


In [4]:
def load_passages(path: Path, n: int) -> tuple[list[str], list[int]]:
    """Return (passage_texts, passage_ids) for the first ``n`` passages."""
    df = pd.read_parquet(path)
    subset = df.head(n)
    return subset["passage"].tolist(), subset.index.tolist()


def load_questions(path: Path, ids: list[int]) -> list[tuple[int, str]]:
    """Return [(question_id, question_text)] for the requested test rows."""
    df = pd.read_parquet(path)
    rows = df.loc[ids]
    return [(int(idx), row["question"]) for idx, row in rows.iterrows()]


def preview(text: str, limit: int = PREVIEW) -> str:
    """Flatten a passage for one-line printing."""
    flat = text.replace("\n", " ")
    return flat[:limit] + ("..." if len(flat) > limit else "")


def passage_lookup(texts: list[str], ids: list[int]) -> dict[int, str]:
    """Map passage id -> text, for turning a hit id back into printable text."""
    return dict(zip(ids, texts))


def mean_pairwise_cosine_distance(
    docs: list[Document], id2vec: dict[int, list[float]]
) -> float:
    """Mean 1 - cosine over every pair of returned passage vectors.

    BGE embeddings are L2-normalized, so cosine similarity is a plain dot
    product and 1 - cosine is a proper distance in [0, 2]: 0.0 means the two
    passages are identical, larger values mean the set is more spread out.
    This is the lab's diversity proxy.
    """
    vecs = [id2vec[d.metadata.get("id")] for d in docs]
    total, n = 0.0, 0
    for i in range(len(vecs)):
        for j in range(i + 1, len(vecs)):
            cos = sum(a * b for a, b in zip(vecs[i], vecs[j]))
            total += 1.0 - cos
            n += 1
    return total / n if n else 0.0


## 3 · Experiment — embed, index, query plain + MMR at every lambda

**WHAT:** `run_experiment` embeds the 100-passage subset once, builds the
FAISS store, runs plain top-5 and MMR top-5 at all three lambdas for each
question, and computes both measures per result set — all into one artifact
dict for the demo and the gate to share.

**WHY:** The gate's monotonicity checks (overlap rises as lambda rises,
spread rises as lambda drops) only make sense if every lambda runs on the
same index and the same plain baseline — which one shared `exp` guarantees.


In [5]:
def run_experiment() -> dict:
    passage_texts, passage_ids = load_passages(PASSAGES_PATH, N_PASSAGES)
    questions = load_questions(TEST_PATH, QUESTION_IDS)

    # --- Embed the whole subset once (batched) + each question once ---------
    embedder = BGEEmbedding(model_name=BGE_MODEL_NAME)
    t0 = time.perf_counter()
    passage_vecs = embedder.embed_documents(passage_texts)
    embed_s = time.perf_counter() - t0

    question_texts = [qtext for _, qtext in questions]
    query_vecs = [embedder.embed_query(q) for q in question_texts]

    # --- Build the FAISS index (in-memory) ----------------------------------
    chunks = [
        Document(page_content=t, metadata={"id": pid})
        for t, pid in zip(passage_texts, passage_ids)
    ]
    # The embedder is attached so MMRRetriever.retrieve() can embed the
    # question via store.embed_query(); add() still uses the precomputed
    # passage vectors (embeddings=... takes precedence).
    store = FAISSVectorStore(embedding=embedder)
    t0 = time.perf_counter()
    store.add(chunks, embeddings=passage_vecs)
    index_s = time.perf_counter() - t0

    id2vec = dict(zip(passage_ids, passage_vecs))

    # --- Per question: plain top-K + MMR at every lambda --------------------
    results = []
    for qi, (qid, qtext) in enumerate(questions):
        plain = store.query_with_scores(query_vecs[qi], top_k=TOP_K)
        plain_ids = [d.metadata.get("id") for d, _ in plain]
        plain_set = set(plain_ids)

        mmr: dict[float, list[Document]] = {}
        overlap: dict[float, int] = {}
        dist: dict[float, float] = {}
        for lam in LAMBDAS:
            retriever = MMRRetriever(store, top_k=MMR_K, lambda_mult=lam)
            docs = retriever.retrieve(qtext)
            mmr[lam] = docs
            overlap[lam] = len(plain_set & {d.metadata.get("id") for d in docs})
            dist[lam] = mean_pairwise_cosine_distance(docs, id2vec)

        results.append(
            {
                "qid": qid,
                "qtext": qtext,
                "plain": plain,
                "plain_ids": plain_ids,
                "plain_dist": mean_pairwise_cosine_distance(
                    [d for d, _ in plain], id2vec
                ),
                "mmr": mmr,
                "overlap": overlap,
                "dist": dist,
            }
        )

    return {
        "passage_texts": passage_texts,
        "passage_ids": passage_ids,
        "questions": questions,
        "embed_s": embed_s,
        "index_s": index_s,
        "results": results,
        "dim": len(passage_vecs[0]),
        "indexed": len(passage_vecs),
    }


## 4 · Run — execute the experiment

**WHAT:** Calls `run_experiment()` — one embed, one index build, all
retrievals — and keeps the artifact dict as `exp`.

**WHY:** Everything after this cell (the demo and the verification gate)
reads from this single `exp`, so the printed numbers and the verified
numbers are guaranteed to come from the same run.


In [6]:
exp = run_experiment()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 5 · Demo — read the artifact

**WHAT:** `print_demo` prints the tradeoff table: per question, the plain
top-5's ids and spread, then MMR at each lambda with its ids, overlap with
plain, and spread. Below it, passage previews for the first question so you
can see *which* passages MMR swapped in.

**WHY:** Read the overlap column as the relevance you give up and the spread
column as the diversity you buy. lambda=1.0 must reproduce the plain ranking
exactly (overlap 5/5); as lambda drops, the set drifts — and the previews
show the near-duplicates being replaced by spread-out passages.


In [7]:
def print_demo(exp: dict) -> None:
    passage_lk = passage_lookup(exp["passage_texts"], exp["passage_ids"])

    print("=" * 66)
    print("Lab 02 — MMR: trading relevance for diversity")
    print(f"{BGE_MODEL_NAME} | FAISS flat-L2 | in-memory only")
    print("=" * 66)

    print(f"\n[1] Corpus (deterministic subset, no randomness):")
    print(f"    {exp['indexed']} passages (first {N_PASSAGES} of 3200, ids {exp['passage_ids'][0]}..{exp['passage_ids'][-1]})")
    print(f"    {len(exp['questions'])} questions from test.parquet:")
    for qid, qtext in exp["questions"]:
        print(f"      [{qid}] {qtext}")

    print(f"\n[2] Embed + index:")
    print(f"    embedded {exp['indexed']} passages in {exp['embed_s']:.2f}s (dim {exp['dim']})")
    print(f"    FAISS index built in {exp['index_s']:.3f}s")

    print(f"\n[3] The tradeoff at a glance — plain top-{TOP_K} vs MMR at each lambda:")
    print(f"    overlap = |plain set ∩ mmr set| (relevance proxy, higher = closer to plain)")
    print(f"    spread  = mean pairwise 1-cosine distance of the returned set (diversity proxy)")
    for r in exp["results"]:
        print(f'\n    Q[{r["qid"]}] "{r["qtext"]}"')
        print(f"      plain top-{TOP_K}: ids={r['plain_ids']}  spread={r['plain_dist']:.4f}")
        for lam in LAMBDAS:
            ids = [d.metadata.get("id") for d in r["mmr"][lam]]
            print(f"      mmr @{lam:<3} : ids={ids}  overlap={r['overlap'][lam]}/{TOP_K}  spread={r['dist'][lam]:.4f}")

    print(f"\n[4] Passage previews — Q[{exp['results'][0]['qid']}] "
          f"\"{exp['results'][0]['qtext']}\" (score = squared L2, LOWER = more similar):")
    r0 = exp["results"][0]
    print(f'\n    plain top-{TOP_K}:')
    for doc, score in r0["plain"]:
        pid = doc.metadata.get("id", "?")
        print(f"      {score:8.4f}  [passage {pid}] {preview(doc.page_content)}")
    for lam in LAMBDAS:
        print(f"\n    MMR lambda_mult={lam} (k={MMR_K}):")
        for rank, doc in enumerate(r0["mmr"][lam], 1):
            pid = doc.metadata.get("id", "?")
            print(f"      {rank}. [passage {pid}] {preview(doc.page_content)}")
    print("      ^ lambda 1.0 keeps the plain ranking; as lambda drops, MMR")
    print("        swaps near-duplicate passages for spread-out ones.")

    print("\n[5] Takeaway")
    print("    MMR is a single knob on the same retriever: lambda_mult=1.0 is")
    print("    plain similarity, 0.0 is farthest-point diversity, and 0.5 sits")
    print("    in between. The overlap column shows the relevance you give up;")
    print("    the spread column shows the diversity you buy. For a RAG context")
    print("    window, a diverse top-5 often beats five near-duplicates of the")
    print("    same fact — but pure diversity (0.0) can drift off-topic, which")
    print("    is why 0.5 is the usual default.")


In [8]:
print_demo(exp)


Lab 02 — MMR: trading relevance for diversity
BAAI/bge-base-en-v1.5 | FAISS flat-L2 | in-memory only

[1] Corpus (deterministic subset, no randomness):
    100 passages (first 100 of 3200, ids 0..99)
    3 questions from test.parquet:
      [1606] Is Uruguay's capital Montevideo?
      [1610] Who founded Montevideo?
      [1604] Is Uruguay located in the northwesten part of Africa?

[2] Embed + index:
    embedded 100 passages in 17.30s (dim 768)
    FAISS index built in 0.098s

[3] The tradeoff at a glance — plain top-5 vs MMR at each lambda:
    overlap = |plain set ∩ mmr set| (relevance proxy, higher = closer to plain)
    spread  = mean pairwise 1-cosine distance of the returned set (diversity proxy)

    Q[1606] "Is Uruguay's capital Montevideo?"
      plain top-5: ids=[36, 15, 64, 2, 0]  spread=0.2686
      mmr @1.0 : ids=[36, 15, 64, 2, 0]  overlap=5/5  spread=0.2686
      mmr @0.5 : ids=[36, 15, 18, 26, 6]  overlap=2/5  spread=0.3716
      mmr @0.0 : ids=[36, 18, 26, 13, 69]  o

## 6 · Verification gate — the same checks the .py runs

**WHAT:** Runs the exact `verify_gate`: dimension/count, exactly `TOP_K`
plain hits, MMR returning `MMR_K` *distinct* docs at every lambda, MMR@1.0
top-1 matching plain top-1 (pure relevance), overlap strictly increasing
0.0 < 0.5 < 1.0, spread(0.0) > spread(1.0) (more diverse), and the three
content checks on each question's plain top-1.

**WHY:** `python 02-mmr.py --verify` must print 11/11 PASS; this cell proves
the notebook reproduces the verified `.py` exactly.


In [9]:
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []

    # Dimension and count match the model / subset.
    checks.append(("embedding dimension is 768 (BGE base)", exp["dim"] == BGE_DIM))
    checks.append((f"exactly {N_PASSAGES} passages indexed", exp["indexed"] == N_PASSAGES))

    # Plain retrieval returns exactly TOP_K scored hits per question.
    checks.append(
        ("each question returns exactly TOP_K plain hits",
         all(len(r["plain"]) == TOP_K for r in exp["results"]))
    )

    # MMR at every lambda returns exactly MMR_K distinct documents (no dupes).
    mmr_distinct = all(
        len(docs) == MMR_K and len({d.metadata.get("id") for d in docs}) == MMR_K
        for r in exp["results"]
        for docs in r["mmr"].values()
    )
    checks.append((f"MMR at every lambda returns {MMR_K} distinct docs", mmr_distinct))

    # lambda_mult=1.0 is pure relevance: it must reproduce the plain
    # similarity ranking (BGE vectors are L2-normalized, so cosine and
    # squared-L2 order identically). Verified empirically: exact match.
    top1_agree = all(
        r["plain_ids"][0] == r["mmr"][1.0][0].metadata.get("id")
        for r in exp["results"]
    )
    checks.append(("MMR lambda=1.0 top-1 == plain top-1 (pure relevance)", top1_agree))

    # Diversity grows as lambda drops: the MMR set drifts away from the plain
    # set. Verified empirically: overlap 1 < 2 < 5 for every question.
    overlap_mono = all(
        r["overlap"][0.0] < r["overlap"][1.0] for r in exp["results"]
    )
    checks.append(("overlap(plain, MMR@0.0) < overlap(plain, MMR@1.0) for every question", overlap_mono))

    overlap_mid = all(
        r["overlap"][0.0] < r["overlap"][0.5] < r["overlap"][1.0]
        for r in exp["results"]
    )
    checks.append(("overlap at lambda=0.5 sits strictly between 0.0 and 1.0", overlap_mid))

    # The diversity proxy moves the other way: lower lambda -> more spread.
    # Verified empirically: spread(0.0) > spread(1.0) for every question.
    dist_mono = all(
        r["dist"][0.0] > r["dist"][1.0] for r in exp["results"]
    )
    checks.append(("mean pairwise distance MMR@0.0 > MMR@1.0 (more diverse)", dist_mono))

    # Content checks: each question's plain top-1 is on-topic.
    q1606_top = exp["results"][0]["plain"][0][0].page_content.lower()
    checks.append(("Q1606 top-1 mentions Montevideo", "montevideo" in q1606_top))
    q1610_top = exp["results"][1]["plain"][0][0].page_content.lower()
    checks.append(("Q1610 top-1 names the Spanish founder of Montevideo", "spanish" in q1610_top))
    q1604_top = exp["results"][2]["plain"][0][0].page_content.lower()
    checks.append(("Q1604 top-1 mentions Uruguay", "uruguay" in q1604_top))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


In [10]:
verify_gate(exp)


verification gate:
  [PASS] embedding dimension is 768 (BGE base)
  [PASS] exactly 100 passages indexed
  [PASS] each question returns exactly TOP_K plain hits
  [PASS] MMR at every lambda returns 5 distinct docs
  [PASS] MMR lambda=1.0 top-1 == plain top-1 (pure relevance)
  [PASS] overlap(plain, MMR@0.0) < overlap(plain, MMR@1.0) for every question
  [PASS] overlap at lambda=0.5 sits strictly between 0.0 and 1.0
  [PASS] mean pairwise distance MMR@0.0 > MMR@1.0 (more diverse)
  [PASS] Q1606 top-1 mentions Montevideo
  [PASS] Q1610 top-1 names the Spanish founder of Montevideo
  [PASS] Q1604 top-1 mentions Uruguay


0